# AISC DeepFake — Validation Prediction Generator

Bu notebook, Learnable Logistic Fusion için gereken **validation prediction CSV'lerini**
üretmek amacıyla hazırlanmıştır.

Hedef 9 çıktı:

- Swin V2 Tiny: Eye / Brow / Mouth
- EfficientNet-B0: Eye / Brow / Mouth
- Swin V2 + Texture Fusion: Eye / Brow / Mouth

## Güvenlik kuralları

- Base modeller **yeniden eğitilmez**.
- Kaynak ROI, checkpoint ve mevcut prediction dosyaları salt okunur.
- Yalnız validation split üzerinde inference yapılır.
- Test verisi kullanılmaz.
- `best.ckpt` tercih edilir.
- Texture scaler yalnız train splitinden yeniden fit edilir.
- Her çıktı ayrı klasöre atomik biçimde yazılır.
- Eksik veya uyumsuz deneyde notebook tahmin yürütmez; ilgili kombinasyonu FAILED olarak raporlar.

In [1]:
# ============================================================
# 1) COLAB SETUP + CONFIG
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

!pip -q install "torch>=2.3" "torchvision>=0.18" "scikit-image>=0.24" \
    "PyWavelets>=1.6" "scikit-learn>=1.4" "pandas>=2.0" "numpy>=1.26" \
    "Pillow>=10.0" "PyYAML>=6.0" "tqdm>=4.66" "joblib>=1.3"

from pathlib import Path
from datetime import datetime, timezone

import os, io, re, json, math, random, hashlib, warnings
import numpy as np
import pandas as pd
import yaml
import joblib

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

ROOT = Path("/content/drive/MyDrive/AISC DeepFake Çalışmaları/Deney 1")

RESULT_ROOTS = {
    "eye": ROOT / "Kader/Deney 1/Sonuçlar",
    "brow": ROOT / "Nazlıcan/Deney 1/Sonuçlar",
    "mouth": ROOT / "Dilara/Deney 1/Sonuçlar",
}

EXPERIMENTS = {
    "swinv2_tiny": {
        "eye": RESULT_ROOTS["eye"] / "20260807_1031_eye_swinv2_tiny_seed42",
        "brow": RESULT_ROOTS["brow"] / "Kas_SwinV2_Tiny_Detayli_Sonuc_pdf",
        "mouth": RESULT_ROOTS["mouth"] / "20260807_2235_mouth_swinv2_tiny_seed42",
    },
    "efficientnet_b0": {
        "eye": RESULT_ROOTS["eye"] / "20260808_0803_eye_efficientnet_b0_seed42",
        "brow": RESULT_ROOTS["brow"] / "20260808_1248_eyebrow_efficientnet_b0_seed42",
        "mouth": RESULT_ROOTS["mouth"] / "20260808_1257_mouth_efficientnet_b0_seed42",
    },
    "swinv2_texture": {
        "eye": RESULT_ROOTS["eye"] / "20260806_1748_eye_swinv2_texturefusion_seed42/full",
        "brow": RESULT_ROOTS["brow"] / "Swin V2-Tiny + LBP + GLCM + Gabor + Wavelet Fusion",
        "mouth": RESULT_ROOTS["mouth"] / "SwinV2_TextureFusion_Mouth/20260807_1550_mouth_swinv2_texturefusion_seed42/full",
    },
}

OUTPUT_ROOT = (
    RESULT_ROOTS["eye"]
    / "Fusion_Experiments"
    / "_validation_predictions_for_logistic_fusion"
)

RUN_ID = datetime.now(timezone.utc).strftime(
    "%Y%m%d_%H%M%S_validation_prediction_export_seed42"
)

RUN_DIR = OUTPUT_ROOT / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=False)

print("RUN:", RUN_DIR)

Mounted at /content/drive
RUN: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deney 1/Kader/Deney 1/Sonuçlar/Fusion_Experiments/_validation_predictions_for_logistic_fusion/20260809_173051_validation_prediction_export_seed42


In [2]:
# ============================================================
# 2) REPRODUCIBILITY + ATOMIC I/O
# ============================================================

import torch
import torch.nn as nn

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)


def atomic_csv(df: pd.DataFrame, target: Path) -> None:
    target = Path(target)
    target.parent.mkdir(parents=True, exist_ok=True)
    tmp = target.with_suffix(target.suffix + ".tmp")
    df.to_csv(tmp, index=False)
    check = pd.read_csv(tmp)
    if len(check) != len(df):
        raise RuntimeError(f"CSV verification failed: {target}")
    os.replace(tmp, target)


def atomic_json(payload, target: Path) -> None:
    target = Path(target)
    target.parent.mkdir(parents=True, exist_ok=True)
    tmp = target.with_suffix(target.suffix + ".tmp")
    with tmp.open("w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2, ensure_ascii=False, default=str)
        f.flush()
        os.fsync(f.fileno())
    with tmp.open("r", encoding="utf-8") as f:
        json.load(f)
    os.replace(tmp, target)

DEVICE: cuda


In [3]:
# ============================================================
# 3) DISCOVERY HELPERS
# ============================================================

def load_yaml_if_exists(path: Path):
    if path.is_file():
        with path.open("r", encoding="utf-8") as f:
            obj = yaml.safe_load(f)
        return obj if isinstance(obj, dict) else {}
    return {}


def find_config(exp_root: Path) -> dict:
    candidates = [
        exp_root / "config_resolved.yaml",
        exp_root.parent / "config_resolved.yaml",
    ]
    for p in candidates:
        if p.is_file():
            return load_yaml_if_exists(p)
    return {}


def find_best_checkpoint(exp_root: Path):
    candidates = [
        exp_root / "checkpoints/best.ckpt",
        exp_root / "checkpoints/finetune/best.ckpt",
        exp_root.parent / "checkpoints/best.ckpt",
        exp_root.parent / "checkpoints/finetune/best.ckpt",
    ]
    for p in candidates:
        if p.is_file():
            return p

    # targeted fallback, never climb outside experiment neighborhood
    for base in [exp_root, exp_root.parent]:
        if base.is_dir():
            found = list(base.glob("checkpoints/**/best.ckpt"))
            if found:
                return sorted(found, key=lambda x: len(str(x)))[0]
    return None


def metadata_candidates(exp_root: Path):
    names = [
        "validated_training_manifest.csv",
        "training_manifest.csv",
        "metadata_canonical_ssot.csv",
        "eligible_metadata.csv",
        "eligible_metadata_before_cache.csv",
    ]
    paths = []
    for base in [exp_root, exp_root.parent]:
        for folder in ["audit", "artifacts", ""]:
            for name in names:
                p = base / folder / name if folder else base / name
                paths.append(p)
    return paths


def normalize_split(v):
    s = str(v).strip().lower()
    if s == "validation":
        return "val"
    return s


def find_split_col(df):
    for c in ["split", "dataset_split", "set"]:
        if c in df.columns:
            return c
    return None


def find_label_col(df):
    for c in ["target", "label_int", "label", "y_true", "true_label"]:
        if c in df.columns:
            return c
    return None


def normalize_label(v):
    if pd.isna(v):
        raise ValueError("NaN label")
    if isinstance(v, (int, float, np.integer, np.floating)):
        return int(float(v) >= 0.5)
    s = str(v).strip().lower()
    if s in {"1", "fake", "deepfake", "manipulated"}:
        return 1
    if s in {"0", "real", "genuine", "original"}:
        return 0
    return int(float(s) >= 0.5)


def choose_existing_image_column(df):
    candidates = [
        "resolved_image_path",
        "training_eye_path",
        "combined_eye_path",
        "combined_brow_path",
        "mouth_path",
        "roi_path",
        "image_path",
        "path",
        "output_path",
    ]

    best = None
    best_fraction = -1.0

    for c in candidates:
        if c not in df.columns:
            continue
        sample = df[c].dropna().astype(str).head(200)
        if sample.empty:
            continue
        fraction = float(sample.map(lambda x: Path(x).is_file()).mean())
        if fraction > best_fraction:
            best = c
            best_fraction = fraction

    if best is None or best_fraction < 0.80:
        return None

    return best


def load_experiment_manifest(exp_root: Path):
    for p in metadata_candidates(exp_root):
        if not p.is_file():
            continue
        try:
            df = pd.read_csv(p)
        except Exception:
            continue

        split_col = find_split_col(df)
        label_col = find_label_col(df)
        image_col = choose_existing_image_column(df)

        if split_col and label_col and image_col:
            work = df.copy()
            work[split_col] = work[split_col].map(normalize_split)
            work = work[work[split_col].isin(["train", "val", "test"])].copy()
            if not work.empty:
                work["_label"] = work[label_col].map(normalize_label).astype(int)
                work["_image_path"] = work[image_col].astype(str)
                return work, {
                    "metadata_path": str(p),
                    "split_col": split_col,
                    "label_col": label_col,
                    "image_col": image_col,
                }

    raise FileNotFoundError(
        f"Usable training/validation manifest not found near: {exp_root}"
    )


def get_sample_id(row):
    for c in ["sample_id", "frame_stem", "source_frame", "image_path", "_image_path"]:
        if c in row.index and pd.notna(row[c]):
            return str(row[c])
    return str(row.name)


def get_video_id(row):
    for c in ["video_id", "source_video", "video"]:
        if c in row.index and pd.notna(row[c]):
            return str(row[c])
    return ""


def get_source_frame(row):
    for c in ["source_frame", "relative_frame_path", "frame_stem", "_image_path"]:
        if c in row.index and pd.notna(row[c]):
            return str(row[c])
    return str(row["_image_path"])

In [4]:
# ============================================================
# 4) IMAGE DATASET + TRANSFORMS
# ============================================================

from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import (
    swin_v2_t,
    Swin_V2_T_Weights,
    efficientnet_b0,
    EfficientNet_B0_Weights,
)
from tqdm.auto import tqdm


class SimpleROIDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True).copy()
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = Path(row["_image_path"])
        image = Image.open(path).convert("RGB")
        return {
            "image": self.transform(image),
            "target": torch.tensor(int(row["_label"]), dtype=torch.float32),
            "sample_id": get_sample_id(row),
            "video_id": get_video_id(row),
            "source_frame": get_source_frame(row),
            "image_path": str(path),
        }


def make_eval_transform(family: str, image_size=224):
    if family == "efficientnet_b0":
        weights = EfficientNet_B0_Weights.DEFAULT
    else:
        weights = Swin_V2_T_Weights.DEFAULT

    mean = weights.meta["mean"]
    std = weights.meta["std"]

    return transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std=std),
    ])

In [5]:
# ============================================================
# 5) SWIN V2 TINY + EFFICIENTNET-B0 MODEL DEFINITIONS
# ============================================================

class SwinV2TinyBinaryClassifier(nn.Module):
    def __init__(self, dropout=0.20):
        super().__init__()
        self.backbone = swin_v2_t(weights=None)
        in_features = self.backbone.head.in_features
        self.backbone.head = nn.Identity()
        self.classifier = nn.Sequential(
            nn.LayerNorm(in_features),
            nn.Dropout(p=float(dropout)),
            nn.Linear(in_features, 1),
        )

    def forward(self, x):
        return self.classifier(self.backbone(x)).squeeze(1)


class EfficientNetB0BinaryClassifier(nn.Module):
    def __init__(self, dropout=0.20):
        super().__init__()
        self.backbone = efficientnet_b0(weights=None)
        in_features = self.backbone.classifier[-1].in_features
        self.backbone.classifier = nn.Identity()
        self.classifier = nn.Sequential(
            nn.Dropout(p=float(dropout)),
            nn.Linear(in_features, 1),
        )

    def forward(self, x):
        return self.classifier(self.backbone(x)).squeeze(1)


def checkpoint_state_dict(ckpt_path):
    state = torch.load(ckpt_path, map_location="cpu", weights_only=False)

    for key in ["model_state_dict", "state_dict", "model"]:
        if key in state and isinstance(state[key], dict):
            return state[key], state

    if all(torch.is_tensor(v) for v in state.values()):
        return state, {"raw_state_dict": True}

    raise RuntimeError(f"No model state_dict found in {ckpt_path}")


def strip_prefix_if_needed(state_dict, prefix):
    if state_dict and all(k.startswith(prefix) for k in state_dict):
        return {k[len(prefix):]: v for k, v in state_dict.items()}
    return state_dict


def infer_dropout(config, ckpt_meta, default=0.20):
    paths = [
        ("model", "dropout"),
        ("dropout",),
    ]

    for root in [config, ckpt_meta.get("config", {}) if isinstance(ckpt_meta, dict) else {}]:
        if not isinstance(root, dict):
            continue
        for parts in paths:
            obj = root
            ok = True
            for p in parts:
                if not isinstance(obj, dict) or p not in obj:
                    ok = False
                    break
                obj = obj[p]
            if ok:
                try:
                    return float(obj)
                except Exception:
                    pass
    return float(default)


def build_and_load_simple_model(family, config, ckpt_path):
    state_dict, meta = checkpoint_state_dict(ckpt_path)
    state_dict = strip_prefix_if_needed(state_dict, "model.")

    dropout = infer_dropout(config, meta, default=0.20)

    if family == "swinv2_tiny":
        model = SwinV2TinyBinaryClassifier(dropout=dropout)
    elif family == "efficientnet_b0":
        model = EfficientNetB0BinaryClassifier(dropout=dropout)
    else:
        raise ValueError(f"Unsupported simple family: {family}")

    try:
        model.load_state_dict(state_dict, strict=True)
    except RuntimeError as exc:
        raise RuntimeError(
            f"{family} checkpoint architecture mismatch.\n"
            f"Checkpoint: {ckpt_path}\n"
            f"Original error: {exc}"
        ) from exc

    model = model.to(DEVICE)
    model.eval()
    return model, meta

In [6]:
# ============================================================
# 6) SIMPLE MODEL VALIDATION INFERENCE
# ============================================================

def predict_simple_validation(family, exp_root):
    config = find_config(exp_root)
    manifest, manifest_audit = load_experiment_manifest(exp_root)

    val_df = manifest[manifest[manifest_audit["split_col"]] == "val"].copy()
    if val_df.empty:
        raise RuntimeError(f"No validation rows found: {exp_root}")

    ckpt = find_best_checkpoint(exp_root)
    if ckpt is None:
        raise FileNotFoundError(f"best.ckpt not found: {exp_root}")

    image_size = 224
    for root in [config]:
        if isinstance(root, dict):
            if "image_size" in root:
                image_size = int(root["image_size"])
            elif isinstance(root.get("data"), dict) and "image_size" in root["data"]:
                image_size = int(root["data"]["image_size"])

    dataset = SimpleROIDataset(
        val_df,
        make_eval_transform(family, image_size=image_size),
    )

    loader = DataLoader(
        dataset,
        batch_size=32,
        shuffle=False,
        num_workers=2,
        pin_memory=(DEVICE.type == "cuda"),
    )

    model, ckpt_meta = build_and_load_simple_model(
        family, config, ckpt
    )

    rows = []

    with torch.no_grad():
        for batch in tqdm(loader, desc=f"{family} validation"):
            images = batch["image"].to(DEVICE, non_blocking=True)
            logits = model(images)
            probs = torch.sigmoid(logits).detach().cpu().numpy()

            if not np.isfinite(probs).all():
                raise FloatingPointError("NaN/Inf validation probability")

            targets = batch["target"].numpy()

            for i in range(len(probs)):
                rows.append({
                    "sample_id": batch["sample_id"][i],
                    "video_id": batch["video_id"][i],
                    "source_frame": batch["source_frame"][i],
                    "image_path": batch["image_path"][i],
                    "target": int(targets[i]),
                    "probability_fake": float(probs[i]),
                })

    pred = pd.DataFrame(rows)
    pred["prediction"] = (pred["probability_fake"] >= 0.5).astype(int)
    pred["correct"] = pred["prediction"] == pred["target"]

    return pred, {
        **manifest_audit,
        "checkpoint": str(ckpt),
        "validation_rows": int(len(pred)),
        "model_family": family,
    }

In [7]:
# ============================================================
# 7) TEXTURE FEATURE PIPELINE
# ============================================================

import cv2
import pywt

from skimage.feature import local_binary_pattern, graycomatrix, graycoprops
from skimage.filters import gabor
from sklearn.preprocessing import StandardScaler

TEXTURE_DEFAULTS = {
    "image_size": 224,
    "lbp_radii": [1, 2, 3],
    "lbp_points": [8, 16, 24],
    "glcm_distances": [1, 2, 4],
    "glcm_angles_deg": [0, 45, 90, 135],
    "glcm_levels": 32,
    "gabor_orientations_deg": [0, 30, 60, 90, 120, 150],
    "gabor_frequencies": [0.10, 0.20, 0.30],
    "wavelet": "db2",
    "wavelet_level": 2,
    "hidden_dim": 256,
    "dropout": 0.30,
}


def texture_config(config):
    out = dict(TEXTURE_DEFAULTS)
    if isinstance(config, dict):
        for key in out:
            if key in config:
                out[key] = config[key]
    return out


def safe_entropy(values, bins=64):
    flat = np.asarray(values, dtype=np.float64).ravel()
    hist, _ = np.histogram(flat, bins=bins, density=False)
    probs = hist.astype(np.float64)
    total = probs.sum()
    if total <= 0:
        return 0.0
    probs /= total
    probs = probs[probs > 0]
    return float(-(probs * np.log2(probs)).sum())


def read_gray(path, size):
    image = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if image is None:
        raise ValueError(f"Unreadable image: {path}")
    if image.shape != (size, size):
        image = cv2.resize(image, (size, size), interpolation=cv2.INTER_AREA)
    return image


def extract_texture_vector(path, cfg):
    gray = read_gray(path, int(cfg["image_size"]))
    features = []
    slices = {}
    start = 0

    # LBP
    vals = []
    for radius, points in zip(cfg["lbp_radii"], cfg["lbp_points"]):
        lbp = local_binary_pattern(gray, P=points, R=radius, method="uniform")
        bins = points + 2
        hist, _ = np.histogram(
            lbp.ravel(), bins=np.arange(0, bins + 1), range=(0, bins)
        )
        hist = hist.astype(np.float32)
        hist /= hist.sum() + 1e-8
        vals.extend(hist.tolist())
    arr = np.asarray(vals, dtype=np.float32)
    slices["lbp"] = slice(start, start + len(arr)); start += len(arr)
    features.append(arr)

    # GLCM
    levels = int(cfg["glcm_levels"])
    quantized = np.floor(gray.astype(np.float32) / 256.0 * levels)
    quantized = np.clip(quantized, 0, levels - 1).astype(np.uint8)
    matrix = graycomatrix(
        quantized,
        distances=cfg["glcm_distances"],
        angles=np.deg2rad(cfg["glcm_angles_deg"]),
        levels=levels,
        symmetric=True,
        normed=True,
    )
    vals = []
    for prop in ["contrast","dissimilarity","homogeneity","energy","correlation","ASM"]:
        values = graycoprops(matrix, prop)
        vals.extend(values.ravel().astype(float).tolist())
    arr = np.asarray(vals, dtype=np.float32)
    slices["glcm"] = slice(start, start + len(arr)); start += len(arr)
    features.append(arr)

    # Gabor
    img = gray.astype(np.float32) / 255.0
    vals = []
    for frequency in cfg["gabor_frequencies"]:
        for angle_deg in cfg["gabor_orientations_deg"]:
            real, imag = gabor(img, frequency=frequency, theta=np.deg2rad(angle_deg))
            mag = np.sqrt(real ** 2 + imag ** 2)
            vals.extend([
                float(mag.mean()),
                float(mag.std()),
                float(np.mean(mag ** 2)),
                safe_entropy(mag),
            ])
    arr = np.asarray(vals, dtype=np.float32)
    slices["gabor"] = slice(start, start + len(arr)); start += len(arr)
    features.append(arr)

    # Wavelet
    coeffs = pywt.wavedec2(
        img,
        wavelet=cfg["wavelet"],
        level=int(cfg["wavelet_level"]),
        mode="symmetric",
    )
    vals = []

    def stats(a):
        a = np.asarray(a, dtype=np.float64)
        aa = np.abs(a)
        return [
            float(a.mean()), float(a.std()), float(np.mean(a ** 2)),
            float(aa.mean()), float(aa.max()), safe_entropy(a),
        ]

    vals.extend(stats(coeffs[0]))
    for horizontal, vertical, diagonal in coeffs[1:]:
        vals.extend(stats(horizontal))
        vals.extend(stats(vertical))
        vals.extend(stats(diagonal))

    arr = np.asarray(vals, dtype=np.float32)
    slices["wavelet"] = slice(start, start + len(arr)); start += len(arr)
    features.append(arr)

    vector = np.concatenate(features).astype(np.float32)
    if not np.isfinite(vector).all():
        raise FloatingPointError(f"NaN/Inf texture vector: {path}")

    return vector, slices

In [8]:
# ============================================================
# 8) SWIN V2 + TEXTURE MODEL + VALIDATION INFERENCE
# ============================================================

class SwinTextureFusion(nn.Module):
    def __init__(self, texture_dim, hidden_dim=256, dropout=0.30):
        super().__init__()

        self.backbone = swin_v2_t(weights=None)
        swin_dim = self.backbone.head.in_features
        self.backbone.head = nn.Identity()

        self.swin_projection = nn.Sequential(
            nn.LayerNorm(swin_dim),
            nn.Linear(swin_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        self.texture_projection = nn.Sequential(
            nn.LayerNorm(texture_dim),
            nn.Linear(texture_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1),
        )

    def forward(self, image, texture):
        deep = self.swin_projection(self.backbone(image))
        tex = self.texture_projection(texture)
        fused = torch.cat([deep, tex], dim=1)
        return self.classifier(fused).squeeze(1)


class TextureDataset(Dataset):
    def __init__(self, df, transform, feature_map, scaler):
        self.df = df.reset_index(drop=True).copy()
        self.transform = transform
        self.feature_map = feature_map
        self.scaler = scaler

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = Path(row["_image_path"])
        image = Image.open(path).convert("RGB")
        sid = get_sample_id(row)
        vec = self.feature_map[sid].reshape(1, -1)
        scaled = self.scaler.transform(vec)[0].astype(np.float32)

        return {
            "image": self.transform(image),
            "texture": torch.from_numpy(scaled),
            "target": torch.tensor(int(row["_label"]), dtype=torch.float32),
            "sample_id": sid,
            "video_id": get_video_id(row),
            "source_frame": get_source_frame(row),
            "image_path": str(path),
        }


def predict_texture_validation(exp_root):
    config = find_config(exp_root)
    cfg = texture_config(config)
    manifest, manifest_audit = load_experiment_manifest(exp_root)

    split_col = manifest_audit["split_col"]
    train_df = manifest[manifest[split_col] == "train"].copy()
    val_df = manifest[manifest[split_col] == "val"].copy()

    if train_df.empty or val_df.empty:
        raise RuntimeError("Texture experiment train/val split is empty.")

    # Recompute deterministic handcrafted features.
    feature_map = {}
    feature_slices = None

    needed = pd.concat([train_df, val_df], ignore_index=True)

    for _, row in tqdm(
        needed.iterrows(),
        total=len(needed),
        desc="Texture features",
    ):
        sid = get_sample_id(row)
        if sid in feature_map:
            continue
        vec, slices = extract_texture_vector(row["_image_path"], cfg)
        feature_map[sid] = vec
        if feature_slices is None:
            feature_slices = slices

    train_matrix = np.stack([
        feature_map[get_sample_id(row)]
        for _, row in train_df.iterrows()
    ])

    scaler = StandardScaler()
    scaler.fit(train_matrix)  # TRAIN ONLY

    texture_dim = int(train_matrix.shape[1])

    ckpt = find_best_checkpoint(exp_root)
    if ckpt is None:
        raise FileNotFoundError(f"best.ckpt not found: {exp_root}")

    state_dict, ckpt_meta = checkpoint_state_dict(ckpt)
    state_dict = strip_prefix_if_needed(state_dict, "model.")

    hidden_dim = int(cfg["hidden_dim"])
    dropout = float(cfg["dropout"])

    # Prefer checkpoint config when available.
    if isinstance(ckpt_meta, dict) and isinstance(ckpt_meta.get("config"), dict):
        c = ckpt_meta["config"]
        hidden_dim = int(c.get("hidden_dim", hidden_dim))
        dropout = float(c.get("dropout", dropout))

    model = SwinTextureFusion(
        texture_dim=texture_dim,
        hidden_dim=hidden_dim,
        dropout=dropout,
    ).to(DEVICE)

    try:
        model.load_state_dict(state_dict, strict=True)
    except RuntimeError as exc:
        raise RuntimeError(
            f"Texture checkpoint architecture mismatch: {ckpt}\n{exc}"
        ) from exc

    model.eval()

    transform = make_eval_transform(
        "swinv2_texture",
        image_size=int(cfg["image_size"]),
    )

    dataset = TextureDataset(
        val_df,
        transform,
        feature_map,
        scaler,
    )

    loader = DataLoader(
        dataset,
        batch_size=16,
        shuffle=False,
        num_workers=2,
        pin_memory=(DEVICE.type == "cuda"),
    )

    rows = []

    with torch.no_grad():
        for batch in tqdm(loader, desc="Texture validation"):
            images = batch["image"].to(DEVICE, non_blocking=True)
            textures = batch["texture"].to(DEVICE, non_blocking=True)
            logits = model(images, textures)
            probs = torch.sigmoid(logits).detach().cpu().numpy()
            targets = batch["target"].numpy()

            if not np.isfinite(probs).all():
                raise FloatingPointError("NaN/Inf texture validation probability")

            for i in range(len(probs)):
                rows.append({
                    "sample_id": batch["sample_id"][i],
                    "video_id": batch["video_id"][i],
                    "source_frame": batch["source_frame"][i],
                    "image_path": batch["image_path"][i],
                    "target": int(targets[i]),
                    "probability_fake": float(probs[i]),
                })

    pred = pd.DataFrame(rows)
    pred["prediction"] = (pred["probability_fake"] >= 0.5).astype(int)
    pred["correct"] = pred["prediction"] == pred["target"]

    return pred, {
        **manifest_audit,
        "checkpoint": str(ckpt),
        "validation_rows": int(len(pred)),
        "texture_dim": texture_dim,
        "hidden_dim": hidden_dim,
        "dropout": dropout,
        "scaler_fit_split": "train_only",
    }

In [9]:
# ============================================================
# 9) RUN ALL 9 VALIDATION EXPORTS
# ============================================================

from sklearn.metrics import roc_auc_score, average_precision_score

run_rows = []

for family, regions in EXPERIMENTS.items():
    for region, exp_root in regions.items():
        print("\n" + "=" * 100)
        print(f"{family} | {region}")
        print("=" * 100)
        print("Experiment:", exp_root)

        output_dir = RUN_DIR / family / region
        output_dir.mkdir(parents=True, exist_ok=True)

        try:
            if not exp_root.is_dir():
                raise FileNotFoundError(f"Experiment directory missing: {exp_root}")

            if family in {"swinv2_tiny", "efficientnet_b0"}:
                pred, audit = predict_simple_validation(family, exp_root)
            elif family == "swinv2_texture":
                pred, audit = predict_texture_validation(exp_root)
            else:
                raise ValueError(f"Unsupported family: {family}")

            if pred.empty:
                raise RuntimeError("Validation prediction output is empty.")

            if pred["target"].nunique() != 2:
                raise RuntimeError("Validation output does not contain both classes.")

            if pred["probability_fake"].isna().any():
                raise RuntimeError("NaN validation probability.")

            if ((pred["probability_fake"] < 0) | (pred["probability_fake"] > 1)).any():
                raise RuntimeError("Probability outside [0,1].")

            roc_auc = float(
                roc_auc_score(pred["target"], pred["probability_fake"])
            )
            pr_auc = float(
                average_precision_score(pred["target"], pred["probability_fake"])
            )

            target_csv = output_dir / "validation_predictions.csv"
            atomic_csv(pred, target_csv)

            atomic_json(
                {
                    **audit,
                    "status": "SUCCESS",
                    "family": family,
                    "region": region,
                    "roc_auc": roc_auc,
                    "pr_auc": pr_auc,
                    "output_csv": str(target_csv),
                    "test_used": False,
                },
                output_dir / "audit.json",
            )

            run_rows.append({
                "family": family,
                "region": region,
                "status": "SUCCESS",
                "n": len(pred),
                "roc_auc": roc_auc,
                "pr_auc": pr_auc,
                "output_csv": str(target_csv),
                "error": "",
            })

            print(f"SUCCESS | n={len(pred)} | ROC-AUC={roc_auc:.4f} | PR-AUC={pr_auc:.4f}")
            print(target_csv)

        except Exception as exc:
            atomic_json(
                {
                    "status": "FAILED",
                    "family": family,
                    "region": region,
                    "experiment_root": str(exp_root),
                    "error_type": type(exc).__name__,
                    "error": str(exc),
                    "test_used": False,
                },
                output_dir / "FAILED.json",
            )

            run_rows.append({
                "family": family,
                "region": region,
                "status": "FAILED",
                "n": 0,
                "roc_auc": np.nan,
                "pr_auc": np.nan,
                "output_csv": "",
                "error": f"{type(exc).__name__}: {exc}",
            })

            print("FAILED:", type(exc).__name__, exc)

summary = pd.DataFrame(run_rows)
atomic_csv(summary, RUN_DIR / "validation_generation_summary.csv")
display(summary)


swinv2_tiny | eye
Experiment: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deney 1/Kader/Deney 1/Sonuçlar/20260807_1031_eye_swinv2_tiny_seed42
FAILED: FileNotFoundError Usable training/validation manifest not found near: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deney 1/Kader/Deney 1/Sonuçlar/20260807_1031_eye_swinv2_tiny_seed42

swinv2_tiny | brow
Experiment: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deney 1/Nazlıcan/Deney 1/Sonuçlar/Kas_SwinV2_Tiny_Detayli_Sonuc_pdf
FAILED: FileNotFoundError Usable training/validation manifest not found near: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deney 1/Nazlıcan/Deney 1/Sonuçlar/Kas_SwinV2_Tiny_Detayli_Sonuc_pdf

swinv2_tiny | mouth
Experiment: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deney 1/Dilara/Deney 1/Sonuçlar/20260807_2235_mouth_swinv2_tiny_seed42
FAILED: FileNotFoundError Usable training/validation manifest not found near: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deney 1/Dilara/Deney 1/Sonuçlar/2

,family,region,status,n,roc_auc,pr_auc,output_csv,error
0,swinv2_tiny,eye,FAILED,0,NaN,NaN,,FileNotFoundError: Usable training/validation ...
1,swinv2_tiny,brow,FAILED,0,NaN,NaN,,FileNotFoundError: Usable training/validation ...
2,swinv2_tiny,mouth,FAILED,0,NaN,NaN,,FileNotFoundError: Usable training/validation ...
3,efficientnet_b0,eye,FAILED,0,NaN,NaN,,FileNotFoundError: Usable training/validation ...
4,efficientnet_b0,brow,FAILED,0,NaN,NaN,,FileNotFoundError: Usable training/validation ...
5,efficientnet_b0,mouth,FAILED,0,NaN,NaN,,FileNotFoundError: Usable training/validation ...
6,swinv2_texture,eye,FAILED,0,NaN,NaN,,FileNotFoundError: Usable training/validation ...
7,swinv2_texture,brow,FAILED,0,NaN,NaN,,FileNotFoundError: Usable training/validation ...
8,swinv2_texture,mouth,FAILED,0,NaN,NaN,,FileNotFoundError: Usable training/validation ...


In [ ]:
# ============================================================
# 10) FINAL QUALITY GATE + LOGISTIC-FUSION PATH MAP
# ============================================================

summary = pd.read_csv(RUN_DIR / "validation_generation_summary.csv")

success_count = int((summary["status"] == "SUCCESS").sum())
failed_count = int((summary["status"] == "FAILED").sum())

print(f"SUCCESS: {success_count}/9")
print(f"FAILED : {failed_count}/9")

path_map = {}

for family in EXPERIMENTS:
    path_map[family] = {}
    for region in ["eye", "brow", "mouth"]:
        row = summary[
            (summary["family"] == family)
            & (summary["region"] == region)
        ]

        if len(row) != 1:
            raise RuntimeError(f"Accounting error: {family}/{region}")

        record = row.iloc[0]

        path_map[family][region] = (
            str(record["output_csv"])
            if record["status"] == "SUCCESS"
            else None
        )

atomic_json(
    {
        "run_id": RUN_ID,
        "success_count": success_count,
        "failed_count": failed_count,
        "all_nine_ready": failed_count == 0,
        "validation_prediction_paths": path_map,
    },
    RUN_DIR / "logistic_fusion_validation_path_map.json",
)

print("\nVALIDATION_PREDICTION_OVERRIDE = {")
for family in path_map:
    print(f'    "{family}": {{')
    for region in path_map[family]:
        value = path_map[family][region]
        print(f'        "{region}": {repr(value)},')
    print("    },")
print("}")

if failed_count == 0:
    print("\n✅ ALL 9 VALIDATION PREDICTION FILES READY.")
else:
    print(
        "\n⚠️ Some exports failed. "
        "Open the corresponding FAILED.json before running Logistic Fusion."
    )

## Sonraki adım

Son hücrenin bastığı `VALIDATION_PREDICTION_OVERRIDE` sözlüğünü
`03_Learnable_Logistic_Fusion_3_Model_Ailesi_REVISED_FROM_SCRATCH.ipynb`
içindeki aynı adlı config alanına yapıştır.

Bu notebook hiçbir TEST prediction dosyasını meta-model eğitimi için kullanmaz.